# Mytho Colab Notebook
Run exploration, smoke tests, optional pretraining, and generation for the Mytho project.

Notes:
- This notebook assumes a GPU runtime.
- For pretraining, you need internet access to stream FineWeb-Edu.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__)

## Setup project path
If you have the repo on GitHub, set `REPO_URL` and keep `REPO_DIR` as a Colab path.
If you uploaded the project manually, set `REPO_DIR` to that folder.

In [ ]:
import os
import sys
import subprocess

REPO_DIR = "/content/Mytho"
REPO_URL = ""  # e.g., https://github.com/you/Mytho.git

if REPO_URL and not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])

if not os.path.exists(REPO_DIR):
    raise FileNotFoundError(
        "Set REPO_DIR to your uploaded project path or set REPO_URL."
    )

os.chdir(REPO_DIR)
print("Repo:", os.getcwd())

## Install minimal dependencies
This installs only the packages needed for data streaming and tokenization.
If you prefer the full set, replace this with `pip install -r requirements.txt`.

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "tiktoken", "datasets"
])

## Exploration: build a small model and run a forward pass

In [ ]:
import torch
from mytho_model import MythoConfig, MythoModel

cfg = MythoConfig(
    vocab_size=1000,
    d_model=256,
    n_heads=4,
    d_head=64,
    d_latent_kv=64,
    d_rope=16,
    n_experts=4,
    n_active_experts=2,
    d_expert_ff=512,
    max_depth=4,
    max_seq_len=64,
)
model = MythoModel(cfg).to(device)
print("Params:", model.num_parameters())

ids = torch.randint(1, cfg.vocab_size, (2, 32), device=device)
out = model(ids, labels=ids)
print("Loss:", float(out["loss"]))
print("Mean depth:", float(out["mean_depth"]))

## Smoke tests
Run the included test scripts.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "test_model.py"])
subprocess.check_call([sys.executable, "test_pretrain.py"])

## Optional: quick pretraining run
Set `RUN_PRETRAIN = True` to execute a short smoke run.

In [ ]:
import subprocess
import sys

RUN_PRETRAIN = False

if RUN_PRETRAIN:
    subprocess.check_call([
        sys.executable, "pretrain.py",
        "--max_docs", "100",
        "--max_steps", "50",
        "--batch_size", "2",
        "--seq_len", "256",
        "--grad_accum", "1"
    ])

## Optional: generation demo
If you have a checkpoint, set `CHECKPOINT_PATH`.
Otherwise it will run with random weights.

In [ ]:
import subprocess
import sys

RUN_GENERATE = False
CHECKPOINT_PATH = ""  # e.g., checkpoints_pretrain/step_1000.pt
PROMPT = "Once upon a time"

if RUN_GENERATE:
    cmd = [sys.executable, "generate.py", "--prompt", PROMPT]
    if CHECKPOINT_PATH:
        cmd += ["--checkpoint", CHECKPOINT_PATH]
    subprocess.check_call(cmd)